In [7]:
#' 绘制四工具Venn图并保存Deep_EccScan唯一结果
#' 
#' @param data_list 包含四个工具数据的列表，顺序为：Deep_EccScan, CircleMap, Circle_finder, ecc_finder
#' @param id_column 用于比较的ID列名，默认为"cluster_id"
#' @param filename 输出文件的基础名，默认为"ggvenn_4tools"
#' @param save_unique_deep_eccscan 是否保存Deep_EccScan独有的结果，默认为TRUE
#' @param save_all_unique 是否保存所有工具的唯一/交集结果，默认为FALSE
#' @return 返回ggvenn图和统计结果列表

draw_ggvenn_4tools <- function(data_list, id_column = "cluster_id", 
                               filename = "ggvenn_4tools",
                               save_unique_deep_eccscan = TRUE,
                               save_all_unique = FALSE) {
  # 加载所需包
  if (!require(ggvenn)) install.packages("ggvenn")
  if (!require(ggsci)) install.packages("ggsci")
  if (!require(ggplot2)) install.packages("ggplot2")
  library(ggvenn)
  library(ggsci)
  library(ggplot2)
  
  # 定义工具名称
  tool_names <- c("Deep_EccScan", "CircleMap", "Circle_finder", "ecc_finder")
  
  # 检查数据是否存在并提取ID列
  venn_data <- list()
  original_data <- list()  # 保存原始数据以便输出
  
  for (i in 1:4) {
    tool_name <- tool_names[i]
    data_df <- data_list[[i]]
    
    if (!is.null(data_df) && nrow(data_df) > 0) {
      # 检查ID列是否存在
      if (id_column %in% names(data_df)) {
        venn_data[[tool_name]] <- unique(data_df[[id_column]])
        original_data[[tool_name]] <- data_df  # 保存原始数据
      } else {
        # 如果指定的列不存在，尝试使用第一列
        cat(paste("警告:", tool_name, "中未找到列", id_column, "，使用第一列替代\n"))
        venn_data[[tool_name]] <- unique(data_df[[1]])
        original_data[[tool_name]] <- data_df
      }
    } else {
      venn_data[[tool_name]] <- character(0)
      original_data[[tool_name]] <- data.frame()
    }
  }
  
  # 检查是否有足够数据
  data_counts <- sapply(venn_data, length)
  if (sum(data_counts > 0) < 2) {
    cat("需要至少2个有数据的工具\n")
    return(NULL)
  }
  
  # 计算总唯一值数量和交集信息
  all_values <- unique(unlist(venn_data))
  total_unique <- length(all_values)
  
  # 1. 计算Deep_EccScan独有的结果
  if (save_unique_deep_eccscan && length(venn_data[["Deep_EccScan"]]) > 0) {
    deep_eccscan_unique <- setdiff(
      venn_data[["Deep_EccScan"]],
      union(
        union(venn_data[["CircleMap"]], venn_data[["Circle_finder"]]),
        venn_data[["ecc_finder"]]
      )
    )
    
    cat(paste("\nDeep_EccScan独有的eccDNA数量:", length(deep_eccscan_unique), "\n"))
    
    if (length(deep_eccscan_unique) > 0) {
      # 提取Deep_EccScan的原始数据
      deep_eccscan_data <- original_data[["Deep_EccScan"]]
      
      # 筛选独有的结果
      if (id_column %in% names(deep_eccscan_data)) {
        unique_data <- deep_eccscan_data[deep_eccscan_data[[id_column]] %in% deep_eccscan_unique, ]
      } else {
        unique_data <- deep_eccscan_data[deep_eccscan_data[[1]] %in% deep_eccscan_unique, ]
      }
      
      # 保存为tsv文件
      tsv_filename <- paste0(filename, "_Deep_EccScan_unique.tsv")
      write.table(unique_data, 
                  file = tsv_filename, 
                  sep = "\t", 
                  row.names = FALSE, 
                  quote = FALSE)
      cat(paste("Deep_EccScan独有的结果已保存到:", tsv_filename, "\n"))
      
      # 同时保存ID列表（可选）
      id_filename <- paste0(filename, "_Deep_EccScan_unique_ids.txt")
      writeLines(deep_eccscan_unique, id_filename)
      cat(paste("Deep_EccScan独有的ID列表已保存到:", id_filename, "\n"))
    } else {
      cat("Deep_EccScan没有独有的eccDNA\n")
    }
  }
  
  # 2. 计算并保存所有工具的交集和独有结果（可选）
  if (save_all_unique) {
    calculate_and_save_overlaps(venn_data, original_data, id_column, filename, tool_names)
  }
  
  # 绘制ggvenn图
  p <- ggvenn(
    venn_data,
    fill_color = ggsci::pal_lancet("lanonc")(4),
    stroke_linetype = "longdash",
    stroke_size = 0.5,
    set_name_size = 6,
    text_size = 5,
    show_percentage = TRUE
  ) +
    labs(
      title = filename,
      subtitle = paste("Total unique eccDNAs:", total_unique)
    ) +
    theme(
      plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 12)
    )
  
  # 保存图像
  ggsave(paste0(filename, ".png"), p, width = 8, height = 8, dpi = 300)
  ggsave(paste0(filename, ".pdf"), p, width = 8, height = 6)
  
  # 输出详细统计信息
  #cat("\n" + strrep("=", 50) + "\n")
  #cat("Venn图详细统计信息:\n")
  #cat(strrep("-", 30) + "\n")
  cat(paste("总唯一eccDNA数量:", total_unique, "\n"))
  cat("\n各工具检测到的数量:\n")
  for (tool in names(venn_data)) {
    cat(sprintf("%-15s : %5d\n", tool, length(venn_data[[tool]])))
  }
  
  # 计算并显示交集信息
  cat("\n工具间重叠统计:\n")
  calculate_overlap_stats(venn_data)
  
  #cat("\n" + strrep("=", 50) + "\n")
  
  # 返回结果列表
  result_list <- list(
    plot = p,
    venn_data = venn_data,
    total_unique = total_unique,
    deep_eccscan_unique = if(exists("deep_eccscan_unique")) deep_eccscan_unique else NULL,
    statistics = list(
      tool_counts = data_counts,
      overlaps = calculate_overlap_matrix(venn_data)
    )
  )
  
  return(p)
}

#' 辅助函数：计算并保存所有重叠结果
calculate_and_save_overlaps <- function(venn_data, original_data, id_column, filename, tool_names) {
  # 创建保存重叠结果的目录
  overlap_dir <- paste0(filename, "_overlap_results")
  if (!dir.exists(overlap_dir)) {
    dir.create(overlap_dir)
  }
  
  # 计算所有可能的组合
  combinations <- list(
    list(name = "All_four_tools", tools = tool_names),
    list(name = "Deep_EccScan_CM_CF", tools = tool_names[1:3]),
    list(name = "Deep_EccScan_CM_EF", tools = c(tool_names[1:2], tool_names[4])),
    list(name = "Deep_EccScan_CF_EF", tools = c(tool_names[1], tool_names[3:4])),
    list(name = "CM_CF_EF", tools = tool_names[2:4]),
    list(name = "Deep_EccScan_CM", tools = tool_names[1:2]),
    list(name = "Deep_EccScan_CF", tools = c(tool_names[1], tool_names[3])),
    list(name = "Deep_EccScan_EF", tools = c(tool_names[1], tool_names[4])),
    list(name = "CM_CF", tools = tool_names[2:3]),
    list(name = "CM_EF", tools = c(tool_names[2], tool_names[4])),
    list(name = "CF_EF", tools = tool_names[3:4])
  )
  
  # 计算并保存每个组合的交集
  for (combo in combinations) {
    tools <- combo$tools
    tool_ids <- lapply(tools, function(tool) venn_data[[tool]])
    
    # 计算交集
    if (length(tools) > 1) {
      intersect_ids <- Reduce(intersect, tool_ids)
    } else {
      intersect_ids <- tool_ids[[1]]
    }
    
    if (length(intersect_ids) > 0) {
      # 保存交集ID列表
      id_file <- paste0(overlap_dir, "/", filename, "_", combo$name, "_ids.txt")
      writeLines(intersect_ids, id_file)
      
      # 尝试从Deep_EccScan中提取完整数据（如果存在）
      if ("Deep_EccScan" %in% tools) {
        deep_data <- original_data[["Deep_EccScan"]]
        if (nrow(deep_data) > 0) {
          if (id_column %in% names(deep_data)) {
            overlap_data <- deep_data[deep_data[[id_column]] %in% intersect_ids, ]
          } else {
            overlap_data <- deep_data[deep_data[[1]] %in% intersect_ids, ]
          }
          
          if (nrow(overlap_data) > 0) {
            data_file <- paste0(overlap_dir, "/", filename, "_", combo$name, ".tsv")
            write.table(overlap_data, data_file, sep = "\t", row.names = FALSE, quote = FALSE)
          }
        }
      }
    }
  }
  
  cat(paste("\n所有重叠结果已保存到目录:", overlap_dir, "\n"))
}

#' 辅助函数：计算重叠统计
calculate_overlap_stats <- function(venn_data) {
  tools <- names(venn_data)
  active_tools <- tools[sapply(venn_data, length) > 0]
  
  if (length(active_tools) >= 2) {
    for (i in 1:(length(active_tools)-1)) {
      for (j in (i+1):length(active_tools)) {
        tool1 <- active_tools[i]
        tool2 <- active_tools[j]
        overlap <- length(intersect(venn_data[[tool1]], venn_data[[tool2]]))
        #total_union <- length(union(venn_data[[tool1]], venn_data[[tool2]]))
        total_union <- length(venn_data[[tool1]])
        overlap_rate <- ifelse(total_union > 0, round(overlap/total_union*100, 1), 0)
        
        cat(sprintf("%-15s 和 %-15s 重叠: %3d (%.1f%%)\n", 
                   tool1, tool2, overlap, overlap_rate))
      }
    }
  }
}

#' 辅助函数：计算重叠矩阵
calculate_overlap_matrix <- function(venn_data) {
  tools <- names(venn_data)
  n <- length(tools)
  overlap_matrix <- matrix(0, nrow = n, ncol = n, dimnames = list(tools, tools))
  
  for (i in 1:n) {
    for (j in 1:n) {
      if (i != j) {
        overlap_matrix[i, j] <- length(intersect(venn_data[[i]], venn_data[[j]]))
      } else {
        overlap_matrix[i, j] <- length(venn_data[[i]])
      }
    }
  }
  
  return(overlap_matrix)
}


In [18]:
setwd('../05.Real_Samples/11.Mouse_Muscle_MusY1')
Deep_eccScan = readr::read_tsv("Deep_eccScan_mapping_results.tsv", skip = 0, comment = "#", col_names = TRUE, progress =FALSE, show_col_types = FALSE)
CircleMap = readr::read_tsv("CircleMap_mapping_results.tsv", skip = 0, comment = "#", col_names = TRUE, progress =FALSE, show_col_types = FALSE)
Circle_finder = readr::read_tsv("Circle_finder_mapping_results.tsv", skip = 0, comment = "#", col_names = TRUE, progress =FALSE, show_col_types = FALSE)
ecc_finder = readr::read_tsv("ecc_finder_mapping_results.tsv", skip = 0, comment = "#", col_names = TRUE, progress =FALSE, show_col_types = FALSE)
head(CircleMap)


merged_id,sample_position,merged_length,all_original_positions
<chr>,<chr>,<dbl>,<chr>
chr1:3113987-3114220:233,chr1:3113988-3114220,233,"chr1:3113987-3114220:233,chr1:3113988-3114220:232"
chr1:4427267-4427454:187,chr1:4427268-4427454,187,"chr1:4427267-4427454:187,chr1:4427268-4427454:186"
chr1:6141088-6141487:399,chr1:6141089-6141486,399,"chr1:6141088-6141485:397,chr1:6141121-6141486:365,chr1:6141115-6141486:371,chr1:6141113-6141486:373,chr1:6141112-6141486:374,chr1:6141108-6141485:377,chr1:6141107-6141486:379,chr1:6141100-6141486:386,chr1:6141099-6141486:387,chr1:6141098-6141486:388,chr1:6141097-6141486:389,chr1:6141093-6141483:390,chr1:6141092-6141486:394,chr1:6141091-6141486:395,chr1:6141090-6141487:397,chr1:6141090-6141486:396,chr1:6141090-6141485:395,chr1:6141090-6141480:390,chr1:6141089-6141484:395,chr1:6141089-6141483:394,chr1:6141089-6141482:393,chr1:6141089-6141481:392,chr1:6141089-6141480:391,chr1:6141089-6141478:389,chr1:6141089-6141476:387,chr1:6141089-6141473:384,chr1:6141089-6141486:397"
chr1:6365310-6366149:839,chr1:6365312-6366149,839,"chr1:6365310-6366127:817,chr1:6365312-6366149:837,chr1:6365311-6366145:834,chr1:6365310-6366149:839,chr1:6365310-6366142:832"
chr1:6484237-6484623:386,chr1:6484238-6484623,386,"chr1:6484237-6484623:386,chr1:6484238-6484623:385"
chr1:6520447-6521234:787,chr1:6520448-6521234,787,"chr1:6520447-6521234:787,chr1:6520448-6521234:786"


In [19]:
data_list <- list(
  Deep_eccScan,
  CircleMap, 
  Circle_finder, ecc_finder
)

HPV_Hela <- draw_ggvenn_4tools(data_list, id_column = "merged_id", filename="HPV_Hela")
Human_Sperm <- draw_ggvenn_4tools(data_list, id_column = "merged_id", filename="Human_Sperm")
Mouse_Muscle <- draw_ggvenn_4tools(data_list, id_column = "merged_id", filename="Mouse_Muscle")
Mouse_Blood <- draw_ggvenn_4tools(data_list, id_column = "merged_id", filename="Mouse_Blood")


Deep_EccScan独有的eccDNA数量: 100 
Deep_EccScan独有的结果已保存到: Human_Hela_Deep_EccScan_unique.tsv 
Deep_EccScan独有的ID列表已保存到: Human_Hela_Deep_EccScan_unique_ids.txt 
总唯一eccDNA数量: 27687 

各工具检测到的数量:
Deep_EccScan    :  7304
CircleMap       :  6895
Circle_finder   : 26984
ecc_finder      :  3403

工具间重叠统计:
Deep_EccScan    和 CircleMap       重叠: 5907 (80.9%)
Deep_EccScan    和 Circle_finder   重叠: 7175 (98.2%)
Deep_EccScan    和 ecc_finder      重叠: 2699 (37.0%)
CircleMap       和 Circle_finder   重叠: 6718 (97.4%)
CircleMap       和 ecc_finder      重叠: 2933 (42.5%)
Circle_finder   和 ecc_finder      重叠: 2948 (10.9%)
